# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Manual IR and Inverted Matrix coding + Markdown documentation (work as teams)
3. **Push to GitHub** *(15 min)* – Teams commit and push initial notebooks. **Make sure to include your names so it is easy to identify the team that developed the Min-Max code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(15 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. **Push to the course GitHub** *(automated)* – Find the last code cell in this notebook, update your team number, and run the cell. It will push your notebook to the repo at `submissions/team#`

## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and Inverted Index.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 👥 Team Members

| Group Member | Student ID | Sections |
|---|---|---|
| Sultan Akinola Atanda | 9114837 | Biword and positional indexes, phrase queries, merging and submission |
| John Buni | 9115726 | Document collection and tokenizer |
| Echezona (Eche) Oji | 9078881 | Normalization pipeline and Boolean retrieval |
| Antonio Sainz | 9072844 | TF-IDF ranking, RAG and AI agents |

Cells we added are marked with 🧩 and the name of the person who wrote them. The professor's cells are unchanged.

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [1]:
import os
import requests
import feedparser

# Example: Download blog posts from a public RSS feed and save as .txt files
def fetch_and_save_blog_posts(rss_url, save_folder, max_posts=20):
    feed = feedparser.parse(rss_url)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title = entry.get('title', 'untitled')
        content = entry.get('summary', '')
        # Clean filename
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{title}\n\n{content}")
        count += 1
    print(f"Saved {count} blog posts to {save_folder}")

# Example RSS feed: Python Software Foundation blog
rss_url = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(rss_url, 'sample_docs', max_posts=20)

Saved 20 blog posts to sample_docs


In [2]:
# Example: Load text files from a folder
import os

def load_documents(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

# Replace 'sample_docs/' with your actual folder
documents = load_documents('sample_docs/')
print(f"Loaded {len(documents)} documents.")


Loaded 20 documents.


In [3]:
# 🧩 John Buni - Document Collection

from pathlib import Path
from bs4 import BeautifulSoup

def load_documents(folder_path):
    documents = []

    # Sort files numerically to maintain stable document IDs
    files = sorted(
        Path(folder_path).glob("blog_*.txt"),
        key=lambda file: int(file.stem.split("_")[-1])
    )

    for file in files:
        text = file.read_text(encoding="utf-8")

        # Remove HTML tags
        clean_text = BeautifulSoup(text, "html.parser").get_text(
            separator=" ", strip=True
        )

        documents.append(clean_text)

    return documents

documents = load_documents("sample_docs")
document_names = [f.name for f in sorted(Path("sample_docs").glob("blog_*.txt"),
                                         key=lambda f: int(f.stem.split("_")[-1]))]
print(f"Loaded {len(documents)} documents.")

Loaded 20 documents.


### 🧩 Talking Point #1 (John Buni): stable document IDs and HTML noise

Our collection is 20 posts from the Python Software Foundation blog, pulled from its RSS feed by the download cell above. Each post is saved as a text file with the title and the body.

The professor's loader uses `os.listdir`, which doesn't return files in a fixed order, so the same post can get a different document ID on another computer. We load the files sorted by their post number instead (`blog_1`, `blog_2`, ...) so every index and every result in this notebook points to the same documents.

The RSS text is also HTML. Without cleaning, tags like `<p>` and `<b>` turn into the tokens `p` and `b`, and `p` ends up in every single document. BeautifulSoup strips the markup before anything gets tokenized.

## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [4]:
import re

def tokenize(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(tokens[:20])  # Preview first 20 tokens


['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to', 'announce', 'that']


In [5]:
# 🧩 John Buni - Tokenizer Implementation

import re

# Original tokenizer for comparison
def original_tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

# Improved tokenizer preserves contractions and version numbers
def tokenize(text):
    text = text.lower().replace("’", "'")
    return re.findall(
        r"\b\d+(?:\.\d+)+\b|[^\W_]+(?:'[^\W_]+)*",
        text
    )

tokens = tokenize(documents[0])
print(tokens[:20])

['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to', 'announce', 'that']


In [6]:
# 🧩 John Buni - Vocabulary Verification

original_vocabulary = {
    token for doc in documents
    for token in original_tokenize(doc)
}

vocabulary = {
    token for doc in documents
    for token in tokenize(doc)
}

print(f"Documents: {len(documents)}")
print(f"Original vocabulary: {len(original_vocabulary)}")
print(f"Improved vocabulary: {len(vocabulary)}")
print(f"Requirement met: {len(vocabulary) > 2000}")

Documents: 20
Original vocabulary: 2434
Improved vocabulary: 2473
Requirement met: True


In [7]:
# 🧩 John Buni - Tokenizer Comparison

test_sentences = [
    "Visit https://www.python.org/downloads/ for Python 3.13.1.",
    "Email help@example.org if you can’t log in.",
    "Don't panic!!! Python 3.12 is ready - really?"
]

for sentence in test_sentences:
    print("Sentence:", sentence)
    print("Original:", original_tokenize(sentence))
    print("Improved:", tokenize(sentence))
    print("-" * 60)

Sentence: Visit https://www.python.org/downloads/ for Python 3.13.1.
Original: ['visit', 'https', 'www', 'python', 'org', 'downloads', 'for', 'python', '3', '13', '1']
Improved: ['visit', 'https', 'www', 'python', 'org', 'downloads', 'for', 'python', '3.13.1']
------------------------------------------------------------
Sentence: Email help@example.org if you can’t log in.
Original: ['email', 'help', 'example', 'org', 'if', 'you', 'can', 't', 'log', 'in']
Improved: ['email', 'help', 'example', 'org', 'if', 'you', "can't", 'log', 'in']
------------------------------------------------------------
Sentence: Don't panic!!! Python 3.12 is ready - really?
Original: ['don', 't', 'panic', 'python', '3', '12', 'is', 'ready', 'really']
Improved: ["don't", 'panic', 'python', '3.12', 'is', 'ready', 'really']
------------------------------------------------------------


### 🧩 Talking Point #2 (John Buni): contractions and version numbers

The professor's tokenizer splits on every apostrophe and dot, so "don't" becomes `don` + `t` and "3.13.1" becomes `3`, `13` and `1`. Ours keeps contractions and dotted version numbers together, which also lets NLTK's stop word list (it contains "don't", "isn't", "won't", ...) remove them properly. URLs and email addresses still get split into parts like `python` and `org`, since those parts are what people actually search for.

After removing the HTML, the professor's tokenizer finds 2,434 unique tokens and ours finds 2,473, so both are above the 2,000-word target. On a Python blog this matters: someone searching for 3.13 should get posts about 3.13, not every post that happens to contain a 3 and a 13.

## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [8]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def normalize_tokens(tokens):
    return [stemmer.stem(t) for t in tokens if t not in stop_words]

# Example: normalize one document
norm_tokens = normalize_tokens(tokens)
print(norm_tokens[:20])


['reopen', 'psf', 'grant', 'program', 'twice', 'yearli', 'fund', 'round', 'python', 'softwar', 'foundat', 'psf', 'excit', 'announc', 'psf', 'grant', 'program', 'reopen', 'restructur', 'schedul']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 🧩 Talking Point #3 (Eche Oji): Porter stemming is affix stripping

Lowercasing makes `Python` and `python` the same term. Stop word removal drops very common words like *the*, *and*, *to*. Porter stemming then strips suffixes using ordered rules that only apply under certain conditions. For example the `-ational -> -ate` rule turns *relational* into *relate*, and a later step removes the final *e*, so we end up with `relat`. A stem doesn't have to be a real word.

Stemming improves recall because different forms of a word become one term, but it can also merge words that mean different things (*organizations* and *organ* both become `organ`), which hurts precision. The query has to go through exactly the same steps as the documents, otherwise nothing will match.

In [9]:
# 🧩 Echezona (Eche) Oji
import pandas as pd
from IPython.display import display

sentence = 'The relational organizations were running and connected to the national board.'
lowercase = sentence.lower()
sentence_tokens = tokenize(sentence)
filtered = [t for t in sentence_tokens if t not in stop_words]
stemmed = normalize_tokens(sentence_tokens)
pipeline_table = pd.DataFrame([
    ('Raw', sentence), ('Lowercase', lowercase),
    ('Tokens', ' | '.join(sentence_tokens)),
    ('Stop words removed', ' | '.join(filtered)),
    ('Stemmed', ' | '.join(stemmed)),
], columns=['Stage', 'Result'])
with pd.option_context('display.max_colwidth', None):
    display(pipeline_table)
display(pd.DataFrame([(w, stemmer.stem(w)) for w in
    ['relational', 'national', 'organizations', 'running', 'connected']],
    columns=['Original word', 'Final Porter stem']))

,Stage,Result
0,Raw,The relational organizations were running and connected to the national board.
1,Lowercase,the relational organizations were running and connected to the national board.
2,Tokens,the | relational | organizations | were | running | and | connected | to | the | national | board
3,Stop words removed,relational | organizations | running | connected | national | board
4,Stemmed,relat | organ | run | connect | nation | board


,Original word,Final Porter stem
0,relational,relat
1,national,nation
2,organizations,organ
3,running,run
4,connected,connect


### 🧩 Eche Oji: reading the table

*the*, *were*, *and* and *to* are removed as stop words, *running* becomes `run` and *connected* becomes `connect`. We keep duplicate tokens at this stage because term frequency needs them later, while the inverted index only records each document once per term.

Removing stop words also throws away their positions. That's fine for Boolean search but it matters for phrase queries, which is covered in the biword and positional section further down.

## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [10]:
from collections import defaultdict

def build_inverted_index(documents):
    index = defaultdict(list)
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        seen = set()
        for token in tokens:
            if token not in seen:
                index[token].append(doc_id)
                seen.add(token)
    return dict(index)

inverted_index = build_inverted_index(documents)
print(dict(list(inverted_index.items())[:10]))  # Preview first 10 terms


{'reopen': [0], 'psf': [0, 1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'grant': [0, 1, 2, 4, 5, 7, 8, 13, 16, 19], 'program': [0, 1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 16, 18, 19], 'twice': [0], 'yearli': [0, 10], 'fund': [0, 1, 2, 3, 7, 13, 18, 19], 'round': [0, 2, 7, 13], 'python': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'softwar': [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 16, 19]}


### 🧩 Eche Oji: posting lists and Boolean search

A posting list contains the sorted, unique, zero-based IDs of documents containing a normalized term. `election` is looked up as `elect`. AND intersects the positive lists; AND NOT subtracts the excluded lists from those matches. These are document-level conditions, not adjacency or phrase conditions.

In [11]:
# 🧩 Echezona (Eche) Oji
for word in ['python', 'board', 'election', 'software']:
    term = normalize_tokens(tokenize(word))[0]
    print(f'{word!r} -> {term!r}: {inverted_index.get(term, [])}')
assert all(ids == sorted(set(ids)) for ids in inverted_index.values())

'python' -> 'python': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
'board' -> 'board': [0, 2, 6, 7, 11, 12, 13, 14, 16, 17, 18, 19]
'election' -> 'elect': [0, 2, 7, 11, 12, 14, 15, 16, 17, 18, 19]
'software' -> 'softwar': [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 16, 19]


### 🧩 Talking Point #4 (Eche Oji): query optimization

For an AND query we start with the shortest posting list, because the result can never be longer than that list. Two sorted lists are intersected with two pointers: move forward on whichever side has the smaller ID and keep an ID when both sides match. That's O(m + n) per merge instead of comparing every pair, and we stop early if the result becomes empty. AND NOT uses the same one-pass walk to subtract the excluded lists.

`boolean_and_not("python board", "election")` means `python AND board AND NOT election`. It isn't a full Boolean parser; every excluded word acts as its own AND NOT.

In [12]:
# 🧩 Echezona (Eche) Oji
def intersect_sorted(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] == right[j]:
            result.append(left[i])
            i += 1
            j += 1
        elif left[i] < right[j]:
            i += 1
        else:
            j += 1
    return result


def subtract_sorted(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] == right[j]:
            i += 1
            j += 1
        elif left[i] < right[j]:
            result.append(left[i])
            i += 1
        else:
            j += 1
    result.extend(left[i:])
    return result


def boolean_and(query, index=None):
    if index is None:
        index = inverted_index
    terms = set(normalize_tokens(tokenize(query)))
    if not terms:
        return []
    postings = sorted((index.get(t, []) for t in terms), key=len)
    result = list(postings[0])
    for posting in postings[1:]:
        if not result:
            break
        result = intersect_sorted(result, posting)
    return result


def boolean_and_not(positive_query, excluded_query, index=None):
    if index is None:
        index = inverted_index
    result = boolean_and(positive_query, index)
    for term in sorted(set(normalize_tokens(tokenize(excluded_query)))):
        if not result:
            break
        result = subtract_sorted(result, index.get(term, []))
    return result

and_results = boolean_and('python board')
and_not_results = boolean_and_not('python board', 'election')
print('python AND board:', and_results)
print('python AND board AND NOT election:', and_not_results)
removed = sorted(set(and_results) - set(and_not_results))
print('Removed because they contain the normalized election term:', removed)
display(pd.DataFrame([
    {'Document ID': d, 'File': document_names[d],
     'Contains election stem': d in inverted_index.get('elect', []),
     'Kept by AND NOT': d in and_not_results}
    for d in and_results]))

python AND board: [0, 2, 6, 7, 11, 12, 13, 14, 16, 17, 18, 19]
python AND board AND NOT election: [6, 13]
Removed because they contain the normalized election term: [0, 2, 7, 11, 12, 14, 16, 17, 18, 19]


,Document ID,File,Contains election stem,Kept by AND NOT
0,0,blog_1.txt,True,False
1,2,blog_3.txt,True,False
2,6,blog_7.txt,False,True
3,7,blog_8.txt,True,False
4,11,blog_12.txt,True,False
5,12,blog_13.txt,True,False
6,13,blog_14.txt,False,True
7,14,blog_15.txt,True,False
8,16,blog_17.txt,True,False
9,17,blog_18.txt,True,False


### 🧩 Eche Oji: checking the results

`python AND board` returns 12 documents. Adding `AND NOT election` keeps only doc 6 (`blog_7.txt`) and doc 13 (`blog_14.txt`). The other 10 all contain the stem `elect`, so *election*, *elections* and *elected* all get them excluded. The table shows every candidate with its file and why it was kept or dropped.

The next cell double-checks the merge functions three ways: against a slow scan of every document, on a small made-up corpus where we know the right answer, and against Python sets on 100 random lists.

In [13]:
# 🧩 Echezona (Eche) Oji
# Direct-scan oracle: does not use posting-list merge functions.
def scan_expected(positive, negative, docs):
    required = set(normalize_tokens(tokenize(positive)))
    excluded = set(normalize_tokens(tokenize(negative)))
    if not required:
        return []
    return [i for i, doc in enumerate(docs)
            if required <= set(normalize_tokens(tokenize(doc)))
            and not excluded.intersection(normalize_tokens(tokenize(doc)))]

for positive, negative in [
    ('python board', ''), ('python board', 'election'),
    ('PYTHON python board', 'elections'), ('zzzzmissingterm', ''),
    ('the and', ''), ('python', 'the'), ('python', 'python'),
    ('python', 'board election'), ('', ''),
]:
    assert boolean_and_not(positive, negative) == scan_expected(positive, negative, documents)

fixture = ['Python board election', 'Python board meeting',
           'Python software', 'Board elections', 'Python boards']
fixture_index = build_inverted_index(fixture)
assert boolean_and('python board', fixture_index) == [0, 1, 4]
assert boolean_and_not('python board', 'election', fixture_index) == [1, 4]
assert boolean_and_not('python', 'board software', fixture_index) == []
assert boolean_and('python', {}) == []
assert intersect_sorted([], [1, 2]) == []
assert subtract_sorted([1, 3, 5], [0, 3, 6]) == [1, 5]
assert subtract_sorted([1, 3], []) == [1, 3]

# Exercise sorted merge logic over many different list shapes.
import random
rng = random.Random(8245)
for _ in range(100):
    a = sorted(rng.sample(range(30), rng.randrange(31)))
    b = sorted(rng.sample(range(30), rng.randrange(31)))
    assert intersect_sorted(a, b) == sorted(set(a) & set(b))
    assert subtract_sorted(a, b) == sorted(set(a) - set(b))
print('PASS: 9 corpus queries, controlled-corpus cases, edge cases, and 100 merge comparisons.')

PASS: 9 corpus queries, controlled-corpus cases, edge cases, and 100 merge comparisons.


## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries.
- Demonstrate that they return the correct documents.


In [14]:
# Placeholder for phrase query implementation
# You may build a position-aware index or use string search within docs after normalization

# Example:
query1 = "Python software"
query2 = "bylaws"

def phrase_query(query, documents):
    results = []
    query_tokens = normalize_tokens(tokenize(query))
    for doc_id, doc in enumerate(documents):
        doc_tokens = normalize_tokens(tokenize(doc))
        # Search for the sequence in the token list
        for i in range(len(doc_tokens) - len(query_tokens) + 1):
            if doc_tokens[i:i+len(query_tokens)] == query_tokens:
                results.append(doc_id)
                break
    return results

# Run phrase queries on the loaded documents
results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")

Documents containing the phrase 'Python software': [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14]
Documents containing the phrase 'bylaws': [11, 14]


## 🧠 Additional Challenge: Implement Positional Indexes

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons | 
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF)  |  |  |  |
| Inverse Document Frequency (idF weight)  |  |  |  |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [15]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'  # Change to any term you want to inspect
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

Positions for term 'python':
  Document 0: [8, 50, 195, 413, 449, 543, 575, 578, 589, 598, 602, 615, 625]
  Document 1: [84, 92, 467, 478, 489, 498, 502, 515]
  Document 2: [0, 9, 16, 21, 43, 56, 67, 87, 93, 101, 112, 182, 190, 203, 210, 230, 237, 258, 266, 275, 279, 328, 343, 345, 364, 373, 389, 401, 507, 564, 578, 627, 641, 708, 767, 842]
  Document 3: [0, 32, 45, 48, 54, 59, 105, 124, 147, 158, 191, 202, 209, 216, 241, 259, 272, 311, 325, 341, 357, 369, 391, 404, 482]
  Document 4: [6, 80, 145, 147, 150, 161, 170, 174, 187]
  Document 5: [10, 38, 104, 115, 124, 135, 161, 239, 242, 253, 262, 266, 279]
  Document 6: [29, 58, 65, 77, 104, 116, 128, 158, 168, 190, 209, 232, 246, 254, 291, 302, 312, 335, 405, 440, 450, 468, 499, 534, 552, 565, 576, 587, 612, 824, 828, 841]
  Document 7: [15, 25, 121, 126, 143, 163, 307, 329]
  Document 8: [4, 10, 64, 131, 133, 223, 245, 348]
  Document 9: [1, 28, 54, 60, 65, 68, 74, 78, 89, 123, 129, 151]
  Document 10: [0, 11, 17, 30, 45, 60, 95, 109, 1

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

### 🧩 Sultan Akinola Atanda: biword index vs positional index (two phrase query tests)

The professor's `phrase_query` above removes stop words before checking word order, and the positional index cell counts positions after stop words are gone. Both treat words as neighbours even when something was removed between them.

I built two indexes and tested the same two phrase queries on both:

- **Biword index:** every pair of consecutive normalized terms -> the documents containing that pair.
- **Positional index:** every term -> documents -> positions, with positions counted in the original token stream (before stop words are removed), so the gaps are kept.

To check which answers are right, `exact_phrase_docs` does a plain regex search for the phrase in the cleaned text of each document.

In [16]:
# 🧩 Sultan Akinola Atanda
class BiwordIndex:
    def __init__(self, documents):
        self.postings = defaultdict(set)
        for doc_id, text in enumerate(documents):
            terms = normalize_tokens(tokenize(text))
            for pair in zip(terms, terms[1:]):
                self.postings[pair].add(doc_id)

    def pairs(self, phrase):
        terms = normalize_tokens(tokenize(phrase))
        return list(zip(terms, terms[1:]))

    def search(self, phrase):
        pairs = self.pairs(phrase)
        if not pairs:
            return []
        return sorted(set.intersection(*(self.postings.get(pair, set()) for pair in pairs)))


class PositionalIndex:
    def __init__(self, documents):
        self.postings = defaultdict(dict)
        self.content_positions = []
        for doc_id, text in enumerate(documents):
            kept = set()
            for pos, token in enumerate(tokenize(text)):
                if token in stop_words:
                    continue
                self.postings[stemmer.stem(token)].setdefault(doc_id, []).append(pos)
                kept.add(pos)
            self.content_positions.append(kept)

    def offsets(self, phrase):
        terms = [(stemmer.stem(t), pos) for pos, t in enumerate(tokenize(phrase)) if t not in stop_words]
        if not terms:
            return []
        start = terms[0][1]
        return [(term, pos - start) for term, pos in terms]

    def search(self, phrase):
        offsets = self.offsets(phrase)
        if not offsets:
            return {}
        gaps = set(range(offsets[-1][1] + 1)) - {off for _, off in offsets}
        first = offsets[0][0]
        candidates = set.intersection(*(set(self.postings.get(term, {})) for term, _ in offsets))
        hits = {}
        for doc_id in sorted(candidates):
            lookup = {term: set(self.postings[term][doc_id]) for term, _ in offsets}
            starts = [p for p in self.postings[first][doc_id]
                      if all(p + off in lookup[term] for term, off in offsets)
                      and not any(p + gap in self.content_positions[doc_id] for gap in gaps)]
            if starts:
                hits[doc_id] = starts
        return hits


def exact_phrase_docs(phrase):
    pattern = re.compile(r'\b' + r'\W+'.join(map(re.escape, phrase.lower().split())) + r'\b')
    return [doc_id for doc_id, text in enumerate(documents) if pattern.search(text.lower())]


biword_index = BiwordIndex(documents)
phrase_index = PositionalIndex(documents)

print(f"Biword index: {len(biword_index.postings):,} pairs, "
      f"{sum(len(d) for d in biword_index.postings.values()):,} postings")
print(f"Positional index: {len(phrase_index.postings):,} terms, "
      f"{sum(len(p) for d in phrase_index.postings.values() for p in d.values()):,} stored positions")
print("Offsets for 'board of directors':", phrase_index.offsets('board of directors'))

Biword index: 6,170 pairs, 7,066 postings
Positional index: 1,726 terms, 7,835 stored positions
Offsets for 'board of directors': [('board', 0), ('director', 2)]


In [17]:
# 🧩 Sultan Akinola Atanda
phrase_queries = ['board of directors', 'board election candidate']

rows = []
for phrase in phrase_queries:
    biword_docs = biword_index.search(phrase)
    positional_docs = sorted(phrase_index.search(phrase))
    exact_docs = exact_phrase_docs(phrase)
    rows.append({
        'phrase': phrase,
        'biword': biword_docs,
        'positional': positional_docs,
        'exact text match': exact_docs,
        'wrong biword hits': sorted(set(biword_docs) - set(exact_docs)),
        'positional correct': positional_docs == exact_docs,
    })

with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(rows))

,phrase,biword,positional,exact text match,wrong biword hits,positional correct
0,board of directors,"[6, 7, 11, 13, 14, 16, 17, 18, 19]","[6, 11, 13, 14, 16, 17, 18, 19]","[6, 11, 13, 14, 16, 17, 18, 19]",[7],True
1,board election candidate,"[11, 16, 17, 18, 19]","[16, 17, 18, 19]","[16, 17, 18, 19]",[11],True


In [18]:
# 🧩 Sultan Akinola Atanda
def show_context(doc_id, pattern, width=40):
    text = documents[doc_id]
    for match in re.finditer(pattern, text, flags=re.IGNORECASE):
        start, end = max(0, match.start() - width), match.end() + width
        print(f'   doc {doc_id}: ...{text[start:end]}...')

print("'board of directors' -> biword pairs:", biword_index.pairs('board of directors'))
show_context(7, r'board\W+directors')

print("\n'board election candidate' -> biword pairs:", biword_index.pairs('board election candidate'))
tokens_11 = tokenize(documents[11])
for first, second in biword_index.pairs('board election candidate'):
    spot = next(p for p in phrase_index.postings[first][11] if p + 1 in phrase_index.postings[second][11])
    print(f"   doc 11 has ({first}, {second}) at position {spot}: ...{' '.join(tokens_11[spot - 3:spot + 4])}...")
print("   doc 11 positional result:", phrase_index.search('board election candidate').get(11, 'no match'))

doc, starts = next(iter(phrase_index.search('board of directors').items()))
print(f"\nPositional match for 'board of directors' in doc {doc}: board at {starts}, "
      f"director at {[p + 2 for p in starts]}")

'board of directors' -> biword pairs: [('board', 'director')]
   doc 7: ...pate in a text-based live chat with PSF Board Directors. This is a chance to connect, share, an...
   doc 7: ...s of the PSF Staff. The list of the PSF Board Directors can be found on our website. We are pas...

'board election candidate' -> biword pairs: [('board', 'elect'), ('elect', 'candid')]
   doc 11 has (board, elect) at position 4: ...the 2026 psf board election results the...
   doc 11 has (elect, candid) at position 158: ...with all the elected candidates shortly to...
   doc 11 positional result: no match

Positional match for 'board of directors' in doc 6: board at [19], director at [21]


### 🧩 Talking Point #5 (Sultan Akinola Atanda): positional matching, and why we prefer it for bigrams

**How positional matching works.** Each term keeps the position it had in the original token stream, counted before stop words are removed. The query goes through the same steps, so *board of directors* becomes `board` at offset 0 and `director` at offset 2, and offset 1 is a gap that can only be a stop word. A document matches when there's some position *p* with `board` at *p*, `director` at *p + 2*, and no content word at *p + 1*. Only documents that contain every term are checked (the posting lists are intersected first), so we never scan documents that can't match.

**What went wrong with the biword index:**
- *board of directors*: the biword index removes stop words and then pairs neighbours, so "PSF Board Directors" in doc 7 gives the same pair (`board`, `director`) as "Board of Directors". Doc 7 never says "board of directors", but the biword index still returns it.
- *board election candidate*: a 3-word phrase has to be split into two pairs, (`board`, `elect`) and (`elect`, `candid`). Doc 11 contains both pairs but in different places (see the positions printed above), so the biword index returns it. The positional index checks that all three words sit next to each other, so it doesn't.

**Which one we would use for bigrams.** For a plain 2-word search the biword index is slightly faster: one dictionary lookup and no position checks. We still prefer the positional index. The same structure answers single words, bigrams, longer phrases and proximity searches ("within 5 words"), and it gives us term frequency for free (the length of the position list). It also didn't return a single wrong document in our tests. The cost is memory, since it stores every position (compare the sizes printed above). A term-document count matrix can't answer a phrase query at all, because it only knows how often a word appears, not where.

### 📝 Completed Table: Term Frequency (TF) and Inverse Document Frequency (IDF weight)

| Term                           | What is it?                                                                 | How is it used?                          | Pros                                   | Cons                                   |
|-------------------------------|----------------------------------------------------------------------------|------------------------------------------|----------------------------------------|----------------------------------------|
| Term Frequency (TF)            | Number of times a term appears in a document                               | Measures term importance in a document   | Simple, fast, good for ranking         | Doesn't account for common terms       |
| Inverse Document Frequency     | Log ratio of total docs to docs containing the term                        | Weights rare terms higher                | Reduces impact of common words         | Can overweight rare, irrelevant terms  |
| (IDF weight)                   | Used with TF to score term relevance: TF-IDF = TF * IDF                    | Improves search relevance                | Highlights unique, important terms     | Requires global corpus statistics      |

Let's implement the Inverse Document Frequency (IDF)

In [19]:
# Calculate Inverse Document Frequency (IDF) for each term using the positional index
import math

def compute_idf_from_positional_index(positional_index, total_docs):
    idf_scores = {}
    for term, doc_positions in positional_index.items():
        doc_count = len(doc_positions)
        if doc_count > 0:
            # --- IGNORE ---
            # print(f"Computing IDF for {term} '{total_docs}' / {doc_count}")
            idf_scores[term] = math.log(total_docs / doc_count)
        else:
            idf_scores[term] = 0.0
    return idf_scores

# Compute IDF for all terms using the positional index
idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

# Example: Show IDF for a sample term
# NOTE : We applied stemming in the normalization step, so the term may not match exactly if it was stemmed
sample_term = 'softwar'  # Change to any term you want to inspect
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0)}")

IDF for term 'softwar': 0.3566749439387324


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

In [20]:
# Calculate Term Frequency (TF) for the term 'softwar' in each document
# TF is simply the number of times the term appears in a document
import pandas as pd

term = 'softwar'  # Use the stemmed term as before

tf_values = []
for doc_id, doc in enumerate(documents):
    tokens = normalize_tokens(tokenize(doc))
    tf = tokens.count(term)
    tf_values.append(tf)
    print(f"TF for term '{term}' in Document {doc_id}: {tf}")

# Display as a DataFrame for clarity
TF_df = pd.DataFrame({'Document': list(range(len(documents))), 'TF': tf_values})
print(TF_df)

TF for term 'softwar' in Document 0: 3
TF for term 'softwar' in Document 1: 2
TF for term 'softwar' in Document 2: 1
TF for term 'softwar' in Document 3: 6
TF for term 'softwar' in Document 4: 3
TF for term 'softwar' in Document 5: 6
TF for term 'softwar' in Document 6: 3
TF for term 'softwar' in Document 7: 0
TF for term 'softwar' in Document 8: 1
TF for term 'softwar' in Document 9: 1
TF for term 'softwar' in Document 10: 0
TF for term 'softwar' in Document 11: 1
TF for term 'softwar' in Document 12: 0
TF for term 'softwar' in Document 13: 1
TF for term 'softwar' in Document 14: 1
TF for term 'softwar' in Document 15: 0
TF for term 'softwar' in Document 16: 1
TF for term 'softwar' in Document 17: 0
TF for term 'softwar' in Document 18: 0
TF for term 'softwar' in Document 19: 2
    Document  TF
0          0   3
1          1   2
2          2   1
3          3   6
4          4   3
5          5   6
6          6   3
7          7   0
8          8   1
9          9   1
10        10   0
11    

#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.

---
## 🧩 Antonio Sainz: TF-IDF Ranking, RAG and AI Agents

This section takes the index we built above and turns it into a **search engine that ranks results**, then shows how that same search engine feeds an AI model (**RAG**) and how an **AI agent** would use it as a tool.

| Part | What it shows |
|---|---|
| 5.1 | How the TF-IDF score of a document is calculated for a query |
| 5.2 | Ranking chart, and why the top document ranks first |
| 5.3 | `retrieve(query, k=3)`: returns the top passages by TF-IDF score |
| 5.4 | A RAG prompt built from the retrieved passages + the user's question |
| 5.5 | How an AI agent calls `retrieve()` as a tool |
| 5.6 | Talking Point #6: TF-IDF only matches exact words, so real RAG also uses embeddings |

### 🧩 Antonio Sainz: 5.1 How the TF-IDF score is calculated

We reuse the definitions from the cells above:

- **TF** (term frequency) $= \text{tf}(t, d)$: how many times the stemmed term $t$ appears in document $d$.
- **IDF** (inverse document frequency) $= \log\left(\frac{N}{n_t}\right)$: $N$ is the number of documents and $n_t$ the number of documents that contain $t$. A word in every document gets $\log(1) = 0$, so it adds nothing to the ranking.

A query has several words, so the **score of a document for a query** is the sum of the TF-IDF of each query term:

$$
\text{score}(q, d) = \sum_{t \in q} \text{tf}(t, d) \times \text{IDF}(t)
$$

The query goes through the **same pipeline as the documents** (tokenize → remove stop words → Porter stem). Otherwise `"securing"` would never match the stored term `secur`.

In [21]:
# 🧩 Antonio Sainz - 5.1 TF-IDF scoring for a whole query
import math
from collections import Counter
import pandas as pd

N_DOCS = len(documents)

# Term counts per document, using the team's tokenizer and normalization pipeline
doc_term_counts = [Counter(normalize_tokens(tokenize(doc))) for doc in documents]

# IDF from the team's positional index: log(N / number of documents containing the term)
idf = compute_idf_from_positional_index(positional_index, N_DOCS)

def tfidf_breakdown(query):
    """One row per (document, query term): tf, idf and their product."""
    query_terms = normalize_tokens(tokenize(query))
    rows = []
    for doc_id, counts in enumerate(doc_term_counts):
        for term in query_terms:
            tf = counts.get(term, 0)
            term_idf = idf.get(term, 0.0)
            rows.append({'doc_id': doc_id, 'term': term, 'tf': tf,
                         'idf': round(term_idf, 4), 'tf_idf': round(tf * term_idf, 4)})
    return pd.DataFrame(rows)

def rank_documents(query, top_n=10):
    """Documents sorted by total TF-IDF score for the query (only scores > 0)."""
    breakdown = tfidf_breakdown(query)
    if breakdown.empty:
        return pd.DataFrame(columns=['doc_id', 'score', 'title'])
    scores = breakdown.groupby('doc_id', as_index=False)['tf_idf'].sum().rename(columns={'tf_idf': 'score'})
    scores = scores[scores['score'] > 0].sort_values('score', ascending=False).head(top_n)
    scores['title'] = scores['doc_id'].apply(lambda i: documents[i].split('\n')[0][:70])
    return scores.reset_index(drop=True)

demo_query = "python security vulnerability"
print("Query:", demo_query)
print("Normalized query terms:", normalize_tokens(tokenize(demo_query)))
rank_documents(demo_query)

Query: python security vulnerability
Normalized query terms: ['python', 'secur', 'vulner']


,doc_id,score,title
0,5,18.4029,Announcing new Security Developer at the PSF W...
1,2,12.0149,Python is for everyone: Join in the PSF year-e...
2,3,9.5945,How the Python ecosystem responded to the eMud...
3,1,3.9925,De-fanging dangerous triggers in GitHub Action...
4,8,3.9925,"Wikis: Relaunched! In February 2026, the Pytho..."
5,13,3.2064,"Announcing the PSF Strategic Plan 2026 In May,..."
6,19,3.1940,Nina Zakharenko: 2026 PSF Board Election Candi...
7,4,0.7985,Joshua Mcshannon Joins the PSF as Infrastructu...
8,18,0.7985,Petr Andreev: 2026 PSF Board Election Candidat...


### 🧩 Antonio Sainz: 5.2 Ranking chart: why the top document ranks first

The chart shows the TF-IDF score of each matching document for the demo query. Each bar is split by query term, so you can see **which word gives each document its score**.

In [22]:
# 🧩 Antonio Sainz - 5.2 Ranking chart (stacked by query term)
import plotly.express as px

ranking = rank_documents(demo_query)
breakdown = tfidf_breakdown(demo_query)
chart_data = breakdown[breakdown['doc_id'].isin(ranking['doc_id'])].copy()
order = [f"Doc {i}" for i in ranking['doc_id']]
chart_data['document'] = chart_data['doc_id'].apply(lambda i: f"Doc {i}")

fig = px.bar(chart_data, x='document', y='tf_idf', color='term',
             category_orders={'document': order},
             title=f'TF-IDF ranking for the query: "{demo_query}"',
             labels={'tf_idf': 'TF-IDF contribution', 'document': 'Document'})
fig.show()

In [23]:
# 🧩 Antonio Sainz - 5.2 Explain the winner: tf x idf per term for the top 3 documents
top3 = ranking.head(3)
for _, row in top3.iterrows():
    print(f"Doc {row.doc_id} | score = {row.score:.3f} | {row.title}")
    detail = breakdown[breakdown['doc_id'] == row.doc_id]
    for _, d in detail.iterrows():
        print(f"    {d.term:<12} tf = {d.tf:<3} x idf = {d.idf:.3f}  ->  {d.tf_idf:.3f}")

if len(top3) > 0:
    winner = breakdown[breakdown['doc_id'] == top3.iloc[0].doc_id]
    best_term = winner.sort_values('tf_idf', ascending=False).iloc[0]
    print(f"\nDoc {top3.iloc[0].doc_id} ranks first mainly because of '{best_term.term}': "
          f"it appears {best_term.tf} times and its idf is {best_term.idf:.3f}.")

Doc 5 | score = 18.403 | Announcing new Security Developer at the PSF We’re excited to announce
    python       tf = 13  x idf = 0.000  ->  0.000
    secur        tf = 17  x idf = 0.798  ->  13.575
    vulner       tf = 3   x idf = 1.609  ->  4.828
Doc 2 | score = 12.015 | Python is for everyone: Join in the PSF year-end fundraiser & membersh
    python       tf = 36  x idf = 0.000  ->  0.000
    secur        tf = 9   x idf = 0.798  ->  7.187
    vulner       tf = 3   x idf = 1.609  ->  4.828
Doc 3 | score = 9.595 | How the Python ecosystem responded to the eMudhra intermediate comprom
    python       tf = 25  x idf = 0.000  ->  0.000
    secur        tf = 10  x idf = 0.798  ->  7.985
    vulner       tf = 1   x idf = 1.609  ->  1.609

Doc 5 ranks first mainly because of 'secur': it appears 17 times and its idf is 0.798.


**Why the top document ranks first:**

1. **It repeats the query words more often (high TF).** Each repetition adds another `idf` to the score, so a post that is *about* security mentions "security" many times.
2. **Its matching words are rare in the collection (high IDF).** A rare word like `vulner` is worth more per occurrence than a word that appears in almost every post, like `python`, whose idf is close to 0.
3. **Common words barely move the ranking.** Because `python` appears in nearly all 20 PSF posts, it barely changes the order. The rare, specific words decide the winner.

> ⚠️ This simple version uses raw counts, so long documents have an advantage (more words → more chances to repeat a term). Production systems normalize by document length (for example with cosine similarity or BM25).

### 🧩 Antonio Sainz: 5.3 `retrieve(query, k=3)`: top passages by TF-IDF

RAG systems do not send whole documents to the AI model. They send short **passages** (chunks), because models have a limited context window and short chunks keep the answer focused.

`retrieve()`:
1. Removes any leftover HTML from each document and splits it into passages of about 3 sentences.
2. Scores every passage with the same TF-IDF formula, using the **corpus IDF** from 5.1.
3. Returns the `k` best passages with their document ID and score.

In [24]:
# 🧩 Antonio Sainz - 5.3 retrieve(query, k=3)
import re
from bs4 import BeautifulSoup

def clean_text(raw):
    """Strip any leftover HTML and collapse whitespace."""
    return re.sub(r'\s+', ' ', BeautifulSoup(raw, 'html.parser').get_text(' ')).strip()

def split_into_passages(text, sentences_per_passage=3):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [' '.join(sentences[i:i + sentences_per_passage])
            for i in range(0, len(sentences), sentences_per_passage)]

# Passage store: (doc_id, passage_text, passage_term_counts)
passages = []
for doc_id, doc in enumerate(documents):
    for p in split_into_passages(clean_text(doc)):
        if p:
            passages.append((doc_id, p, Counter(normalize_tokens(tokenize(p)))))

def retrieve(query, k=3):
    """Return the k passages with the highest TF-IDF score for the query."""
    query_terms = normalize_tokens(tokenize(query))
    scored = []
    for doc_id, text, counts in passages:
        score = sum(counts.get(t, 0) * idf.get(t, 0.0) for t in query_terms)
        if score > 0:
            scored.append({'doc_id': doc_id, 'score': round(score, 3), 'passage': text})
    return sorted(scored, key=lambda r: r['score'], reverse=True)[:k]

print(f"{len(passages)} passages indexed from {N_DOCS} documents\n")
for r in retrieve(demo_query, k=3):
    print(f"[Doc {r['doc_id']} | score {r['score']}] {r['passage'][:300]}\n")

205 passages indexed from 20 documents

[Doc 5 | score 5.627] Brett’s addition to the PSF positions us to better meet this growing challenge. Having Brett on the team will both build capacity for security work the PSF already does and enable new, proactive work: He’ll help us keep pace with critical ongoing work such as vulnerability remediation, providing vul

[Doc 2 | score 5.602] Our new PSF Strategy page is its permanent home, where we’ll publish annual review findings and updates so the community can follow our progress. Sustaining the Python Security Response Team: As the number of vulnerability reports and CVE advisories grew 3-4X, the mostly-volunteer Python Security Re

[Doc 5 | score 5.59] Announcing new Security Developer at the PSF We’re excited to announce that Brett Smith is joining the Python Software Foundation as a Security Developer. Brett will be working closely with Seth Larson, Security Developer in Residence, and Mike Fiedler, PyPI Safety and Security Engineer, on

### 🧩 Antonio Sainz: 5.4 Building a RAG prompt

**RAG (Retrieval-Augmented Generation)** = search first, then generate. Instead of answering from memory, the AI model receives the passages we retrieved and is told to answer **only from them** and to cite the document ID. This cuts down on invented answers and makes every answer traceable to a source.

In [25]:
# 🧩 Antonio Sainz - 5.4 RAG prompt: retrieved passages + the user's question
def build_rag_prompt(question, k=3):
    results = retrieve(question, k=k)
    if not results:
        context = "(no relevant passages found)"
    else:
        context = "\n\n".join(f"[Doc {r['doc_id']}] {r['passage']}" for r in results)
    return (
        "You are a helpful assistant. Answer the question using ONLY the context below.\n"
        "Cite the document IDs you used, e.g. [Doc 3]. If the context does not contain "
        "the answer, say you don't know.\n\n"
        f"### Context\n{context}\n\n"
        f"### Question\n{question}\n\n"
        "### Answer\n"
    )

user_question = "What is the PSF doing to improve Python security?"
print(build_rag_prompt(user_question))

You are a helpful assistant. Answer the question using ONLY the context below.
Cite the document IDs you used, e.g. [Doc 3]. If the context does not contain the answer, say you don't know.

### Context
[Doc 5] Announcing new Security Developer at the PSF We’re excited to announce that Brett Smith is joining the Python Software Foundation as a Security Developer. Brett will be working closely with Seth Larson, Security Developer in Residence, and Mike Fiedler, PyPI Safety and Security Engineer, on the PSF’s security team to address the increasingly demanding scale of security response for the Python ecosystem. We’re grateful to Anthropic for their generous support , which has made the critical new Security Developer role possible.

[Doc 5] Security Response at Ecosystem Scale At PyCon US 2026, Seth Larson and Mike Fiedler presented that 2026 was the “year of more” when it comes to security: the Python Security Response Team (PSRT) is processing 3-4× the volume of reports and CVEs for Py

### 🧩 Antonio Sainz: 5.5 How an AI agent uses `retrieve()` as a tool

An **AI agent** is a model that can decide to call functions (tools) before it answers. We describe `retrieve()` to the model with a name, a description and its parameters. The loop then works like this:

1. **User asks** a question.
2. **The model decides** it needs facts it doesn't have, so instead of answering it returns a *tool call*: `retrieve(query="...", k=3)`. The model writes its own search query, often shorter and more keyword-focused than the user's question.
3. **Our code runs** `retrieve()` (the inverted index + TF-IDF from this notebook) and sends the passages back to the model.
4. **The model answers** from those passages and cites them. If the results are poor, it can call the tool again with a different query.

The cell below shows the tool definition (the same JSON-schema format used by OpenAI and Anthropic tool calling) and simulates one round of the loop without calling a real model.

In [26]:
# 🧩 Antonio Sainz - 5.5 Tool definition + one simulated agent step (no real LLM call)
import json

retrieve_tool = {
    "name": "retrieve",
    "description": "Search the PSF blog collection and return the most relevant passages "
                   "ranked by TF-IDF. Use it before answering questions about PSF news.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Keywords to search for"},
            "k": {"type": "integer", "description": "Number of passages to return", "default": 3},
        },
        "required": ["query"],
    },
}
TOOLS = {"retrieve": retrieve}

# Step 2 (simulated): the model replies with a tool call instead of an answer
model_tool_call = {"name": "retrieve", "input": {"query": "security vulnerability", "k": 3}}

# Step 3: our code executes the tool and returns the result to the model
tool_result = TOOLS[model_tool_call["name"]](**model_tool_call["input"])

print("Tool definition sent to the model:\n", json.dumps(retrieve_tool, indent=2))
print("\nModel requested:", model_tool_call)
print(f"\nTool returned {len(tool_result)} passages -> sent back to the model to write the final answer:")
for r in tool_result:
    print(f"  [Doc {r['doc_id']} | score {r['score']}] {r['passage'][:120]}...")

Tool definition sent to the model:
 {
  "name": "retrieve",
  "description": "Search the PSF blog collection and return the most relevant passages ranked by TF-IDF. Use it before answering questions about PSF news.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Keywords to search for"
      },
      "k": {
        "type": "integer",
        "description": "Number of passages to return",
        "default": 3
      }
    },
    "required": [
      "query"
    ]
  }
}

Model requested: {'name': 'retrieve', 'input': {'query': 'security vulnerability', 'k': 3}}

Tool returned 3 passages -> sent back to the model to write the final answer:
  [Doc 5 | score 5.627] Brett’s addition to the PSF positions us to better meet this growing challenge. Having Brett on the team will both build...
  [Doc 2 | score 5.602] Our new PSF Strategy page is its permanent home, where we’ll publish annual review findings and update

### 🧩 Talking Point #6 (Antonio Sainz): TF-IDF only matches exact words

**Issue:** TF-IDF only finds a document when the query shares the **same stemmed word**. It has no idea that *vote* and *ballot*, or *bug* and *vulnerability*, mean similar things. If the user's wording differs from the author's, a relevant passage gets a score of 0 and is never sent to the AI model. In RAG, a missed retrieval means a wrong or "I don't know" answer, even when the information is in the collection.

The cell below runs two queries that mean almost the same thing but use different words, and compares what each one finds.

In [27]:
# 🧩 Antonio Sainz - 5.6 Same meaning, different words -> different (or no) results
# Each pair means (almost) the same thing but shares no word after stemming
query_pairs = [
    ("vote", "ballot"),
    ("bug", "vulnerability"),
    ("meetup", "conference"),
]
for q_a, q_b in query_pairs:
    docs_a = set(rank_documents(q_a, top_n=N_DOCS)['doc_id'])
    docs_b = set(rank_documents(q_b, top_n=N_DOCS)['doc_id'])
    print(f"'{q_a}' -> terms {normalize_tokens(tokenize(q_a))}: {len(docs_a)} docs")
    print(f"'{q_b}' -> terms {normalize_tokens(tokenize(q_b))}: {len(docs_b)} docs")
    print(f"  Overlap: {len(docs_a & docs_b)} docs | only in first: {sorted(docs_a - docs_b)} | only in second: {sorted(docs_b - docs_a)}\n")

'vote' -> terms ['vote']: 5 docs
'ballot' -> terms ['ballot']: 4 docs
  Overlap: 4 docs | only in first: [2] | only in second: []

'bug' -> terms ['bug']: 1 docs
'vulnerability' -> terms ['vulner']: 4 docs
  Overlap: 1 docs | only in first: [] | only in second: [3, 5, 13]

'meetup' -> terms ['meetup']: 2 docs
'conference' -> terms ['confer']: 8 docs
  Overlap: 2 docs | only in first: [] | only in second: [0, 9, 16, 17, 18, 19]



**How real RAG systems fix this:** they add **embeddings**. A neural model turns every passage and the query into a vector of numbers where *meaning* decides the position, so "vote" and "ballot" end up close together even with no shared word. Most production systems use **hybrid search**:

| Method | Strength | Weakness |
|---|---|---|
| TF-IDF / BM25 (keyword) | Exact names, codes, version numbers like `3.13`; fast; easy to explain | Misses synonyms and paraphrases |
| Embeddings (semantic) | Finds passages with the same meaning in different words | Can miss exact rare terms; needs a model and a vector database |
| Hybrid (both, results merged) | Gets the best of both | More components to build and maintain |

**Takeaway:** the inverted index and TF-IDF from this workshop are still the keyword half of modern RAG. Embeddings add the meaning half.

## 🧠 Reflection

- **John:** most of the work in collection was cleaning, not downloading. The HTML tags and the unsorted file order caused problems further down the notebook that were hard to trace back to the loader.
- **Eche:** writing a slow scan to compare against the posting-list merges caught mistakes quickly. Stemming groups word forms nicely, but sometimes it merges words that have nothing to do with each other.
- **Sultan:** the biword index looked correct until we checked it against the raw text. Removing stop words before counting positions makes different phrases look identical, so positions have to come from the original text.
- **Antonio:** TF-IDF ranks well when the query uses the same words as the post, but it misses synonyms completely. That's why real RAG systems mix keyword search with embeddings.
- **As a team:** each of us worked in our own notebook and Sultan merged everything. We agreed early on the shared `sample_docs` folder and the sorted file order so our document IDs matched.


In [28]:
# Combine TF and IDF to compute TF-IDF for the term 'softwar' in each document
# this is where the user left off
def query_processor(query, inverted_index):
    results = []
    for term in query.split():
        if term in inverted_index:
            results.extend(inverted_index[term])
    return list(set(results))  # ✅ return a list



## 🧠 Push your code to the course GitHub repository

Update your team number and then run the code. 

In [ ]:
!python submit_assignment.py --notebook IR_InvertedMatrix_Workshop.ipynb --student_id "team1"
